# 2. One fold, five models, four metrics

Notebook 1 built the splits. This one runs a **single** fold end to end, so the
model APIs and the metrics are clear before the full grid in notebook 3.

Everything here is deliberately small and slow to read. In the real run this is
a loop.

In [1]:
import os, sys, json, time, warnings
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

d = pd.read_csv("data/banking/1_merged/financial_calm_2013_2018.csv") \
      .sort_values("vintage").reset_index(drop=True)
spec = json.load(open("data/banking/2_folds/financial-calm.json"))

fold = spec["temporal"][-1]                 # last time-ordered fold
tr = np.array(fold["train"]); te = np.array(fold["test"])
print(f"train {len(tr):,} loans, vintages {sorted(d.vintage[tr].unique())}")
print(f"test  {len(te):,} loans, vintage  {sorted(d.vintage[te].unique())}")

train 45,834 loans, vintages [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
test  9,166 loans, vintage  [np.int64(2018)]


## 2.1 Features, and where scaling belongs

Scaling is fitted on the **training rows only**, then applied to the test rows.

If you scale the whole dataset first, the mean and standard deviation are
computed using the test data, and every model trains on features that encode
something about its own test set. That is leakage, and it is worst for exactly
the comparison we care about: the time-ordered split exists to keep future
information out of training, and whole-dataset scaling puts it straight back in.

This is why the shared CSV holds raw values and scaling happens here.

In [2]:
NUM = ["fico","ltv","cltv","dti","dti_missing","orig_upb","orig_rate",
       "orig_term","mi_pct","n_units","n_borrowers"]

X = d[NUM].copy()
for c in NUM:
    X[c] = X[c].fillna(X[c].median())          # see note below on dti

mu = X.iloc[tr].mean()                          # TRAIN only
sd = X.iloc[tr].std().replace(0, 1)             # TRAIN only
Xs = ((X - mu) / sd).to_numpy("float64")        # float64: the deep models require it

t = d.time.to_numpy("float64")
e = d.event.to_numpy("int64")
print("feature matrix", Xs.shape, Xs.dtype)
print("events:", dict(zip(*np.unique(e, return_counts=True))))

feature matrix (55000, 11) float64
events: {np.int64(0): np.int64(19458), np.int64(1): np.int64(2184), np.int64(2): np.int64(33358)}


> **On `dti`.** It is imputed here only to keep this notebook short. `dti` is
> missing for 28.8% of 2013 loans and 1.0% of 2018 loans, so a median fill
> computed across vintages is itself a form of leakage. In the real run, impute
> inside the fold (train statistics only) and keep `dti_missing` as a feature —
> whether a lender recorded DTI is informative in its own right.

## 2.2 The horizons

All models are scored at the same six horizons, 12 to 72 months. The
upper bound captures about 87% of all observed events while staying inside the
follow-up available in every cohort, and matches the horizons Blumenstock et
al. use on this dataset. Choosing them in advance matters:
pick them after seeing results and you are selecting the horizons that flatter
your conclusion.

In [3]:
TIMES = np.array([12., 24., 36., 48., 60., 72.])
y_tr = pd.DataFrame({"event": e[tr], "duration": t[tr]})
y_te = pd.DataFrame({"event": e[te], "duration": t[te]})
y_te.head(3)

,event,duration
0,2,28.0
1,2,23.0
2,1,26.0


## 2.3 Each model returns a different shape

This is the main practical trap. All four produce cumulative incidence, but none
of them agrees on the array layout, and none of them errors if you index it
wrong — you just get numbers that look fine and mean nothing.

| model | call | returns |
|---|---|---|
| Fine–Gray | `predict_cumulative_incidence(X, times)` | `(n, times)` — **one cause per fitted model** |
| CR-Forest | `predict_cif(X, times)` | `(n, causes, times)` |
| DeepHit | `predict_cif(x)` | `(causes, times, n)` — **axes transposed** |
| DSM | `predict_risk(x, horizons, risk=k)` | `(n, times)` per cause |

We normalise everything to `F1, F2` of shape `(n, times)` and then pack once.

In [4]:
from comprisk import FineGrayRegression, CompetingRiskForest, CumulativeIncidence, Surv

y_surv_tr = Surv.from_arrays(event=e[tr], time=t[tr])
preds, timing = {}, {}

# --- Fine-Gray: one model per cause -------------------------------------
s = time.perf_counter()
fg1 = FineGrayRegression(cause=1).fit(Xs[tr], y_surv_tr)
fg2 = FineGrayRegression(cause=2).fit(Xs[tr], y_surv_tr)
preds["Fine-Gray"] = (np.asarray(fg1.predict_cumulative_incidence(Xs[te], times=TIMES)),
                      np.asarray(fg2.predict_cumulative_incidence(Xs[te], times=TIMES)))
timing["Fine-Gray"] = time.perf_counter() - s

# --- competing-risks forest: causes on axis 1 ---------------------------
s = time.perf_counter()
rf = CompetingRiskForest(n_estimators=100, random_state=0).fit(Xs[tr], y_surv_tr)
cif = np.asarray(rf.predict_cif(Xs[te], times=TIMES))
preds["CR-Forest"] = (cif[:, 0, :], cif[:, 1, :])
timing["CR-Forest"] = time.perf_counter() - s

for k, (f1, f2) in preds.items():
    print(f"{k:12s} F1 {f1.shape}  F2 {f2.shape}   fit {timing[k]:.1f}s")

Fine-Gray    F1 (9166, 6)  F2 (9166, 6)   fit 0.1s
CR-Forest    F1 (9166, 6)  F2 (9166, 6)   fit 3.5s


### DeepHit needs three extra pieces

1. **Discretised time.** It models a probability per (time-bin, cause), so
   continuous durations must be binned. pycox ships a transform for single-event
   only, so the competing-risks version is written here.
2. **A 3-D network output** `(batch, causes, bins)`. A flat vector raises
   `IndexError` from deep inside pycox.
3. **Its CIFs come back transposed** — `(causes, times, samples)`.

Its prediction times are the bin edges, not our horizons, so we interpolate onto
`TIMES` to keep every model comparable.

In [5]:
import torch, torchtuples as tt
from pycox.models import DeepHit
from pycox.preprocessing.label_transforms import LabTransDiscreteTime

class CRLabTransform(LabTransDiscreteTime):
    """Discretise durations but keep the cause label (pycox's is single-event)."""
    def transform(self, durations, events):
        dur, is_event = super().transform(durations, events > 0)
        ev = np.asarray(events).copy()
        ev[is_event == 0] = 0
        return dur, ev.astype("int64")

NB, NC = 20, int(e.max())
labtrans = CRLabTransform(NB)
yd_tr = labtrans.fit_transform(t[tr], e[tr])

class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.body = tt.practical.MLPVanilla(Xs.shape[1], [32, 32], NC * NB,
                                            batch_norm=True, dropout=0.1)
    def forward(self, x):
        return self.body(x).view(x.size(0), NC, NB)     # (batch, causes, bins)

s = time.perf_counter()
dh = DeepHit(Net(), tt.optim.Adam(0.01), alpha=0.2, sigma=0.1,
             duration_index=labtrans.cuts)
dh.fit(Xs[tr].astype("float32"), yd_tr, batch_size=256, epochs=10, verbose=False)
raw = np.asarray(dh.predict_cif(Xs[te].astype("float32")))   # (causes, bins, n)
timing["DeepHit"] = time.perf_counter() - s

def to_horizons(curve_bins, bin_edges, times):
    """Interpolate a per-bin CIF onto our evaluation horizons."""
    return np.stack([np.interp(times, bin_edges, row) for row in curve_bins])

preds["DeepHit"] = (to_horizons(raw[0].T, labtrans.cuts, TIMES),
                    to_horizons(raw[1].T, labtrans.cuts, TIMES))
print("DeepHit raw", raw.shape, "-> F1", preds["DeepHit"][0].shape,
      f"  fit {timing['DeepHit']:.1f}s")

DeepHit raw (2, 20, 9166) -> F1 (9166, 6)   fit 3.6s


### The baseline

`CumulativeIncidence` is the Aalen–Johansen estimator. It uses **no covariates**,
so it assigns every subject the same curve and cannot discriminate at all — but
it is marginally calibrated by construction.

It is not a competitor. It does three jobs:

- fixes the scale: it tells you what a good calibration score looks like here
- detects bugs: if your concordance code does not score it at chance, the code is wrong
- separates "the model is bad" from "the population shifted"

In [6]:
s = time.perf_counter()
aj = CumulativeIncidence().fit(time=t[tr], event=e[tr])

# curves_ is keyed by (group, cause); group is None when no groups are used.
# Each value is a CIFCurve with .times and .cif, so interpolate onto our
# horizons and give every test subject the same curve -- that is the point.
def aj_curve(cause):
    c = aj.curves_[(None, cause)]
    return np.interp(TIMES, c.times, c.cif)

preds["Aalen-Johansen"] = (np.tile(aj_curve(1), (len(te), 1)),
                           np.tile(aj_curve(2), (len(te), 1)))
timing["Aalen-Johansen"] = time.perf_counter() - s
print("marginal CIF at", TIMES.astype(int), "months")
print("  default ", np.round(aj_curve(1), 4))
print("  prepaid ", np.round(aj_curve(2), 4))
print("identical for every subject -> cannot discriminate, by construction")

marginal CIF at [12 24 36 48 60 72] months
  default  [0.002  0.0063 0.0141 0.0237 0.0313 0.0369]
  prepaid  [0.0562 0.147  0.2625 0.3822 0.4928 0.5798]
identical for every subject -> cannot discriminate, by construction


## 2.4 Pack the array the metrics expect

`hazardous` wants `(n_samples, n_events + 1, n_times)` with **component 0 the
survival function**, not an event. Put a cause there and every metric returns a
plausible, meaningless number.

In [7]:
def pack(f1, f2):
    f1 = np.clip(np.asarray(f1, float), 0, 1)
    f2 = np.clip(np.asarray(f2, float), 0, 1)
    surv = np.clip(1.0 - f1 - f2, 0, 1)
    return np.stack([surv, f1, f2], axis=1)          # (n, 3, n_times)

packed = {k: pack(*v) for k, v in preds.items()}
for k, p in packed.items():
    print(f"{k:12s} {p.shape}   component 0 is survival: {p[:,0,:].max():.3f} max")

Fine-Gray    (9166, 3, 6)   component 0 is survival: 0.978 max
CR-Forest    (9166, 3, 6)   component 0 is survival: 0.988 max
DeepHit      (9166, 3, 6)   component 0 is survival: 0.976 max
Aalen-Johansen (9166, 3, 6)   component 0 is survival: 0.942 max


## 2.5 The four metrics, and what each one can and cannot see

**Concordance** asks only whether the *ordering* is right. It is invariant to any
monotone transform of the predictions — halve every probability and it does not
move. So it can never detect miscalibration.

**Integrated Brier score** is a proper scoring rule. It decomposes into
calibration *plus* discrimination, so it is a summary, not a diagnostic: a model
can win it by ranking well while being badly calibrated.

**AJ-calibration** compares the mean predicted incidence against the
Aalen–Johansen estimate, integrated over time. Marginal, and uncontaminated by
ranking — this is the one that makes SQ2 answerable.

**CR D-calibration** asks whether subjects assigned probability p experience the
event at rate p. Distributional, so it is a stricter test than AJ-calibration. A
model can pass the first and fail the second.

In [8]:
from hazardous.metrics import (concordance_index_incidence, aj_calibration,
                               integrated_brier_score_incidence, d_cr_calibration)
from hazardous.utils import make_time_grid

grid = make_time_grid(event=e[tr], duration=t[tr], n_time_grid_steps=40)

rows = []
for name, p in packed.items():
    f1 = p[:, 1, :]
    c = concordance_index_incidence(y_te, f1, y_train=y_tr, time_grid=TIMES,
                                    taus=list(TIMES), event_of_interest=1)
    c = float(np.mean(c)) if hasattr(c, "__len__") else float(c)
    ibs = float(integrated_brier_score_incidence(y_tr, y_te, f1, TIMES, event_of_interest=1))
    ajc = float(aj_calibration(y_te, TIMES, p))
    rows.append({"model": name, "concordance": round(c, 4),
                 "IBS": round(ibs, 5), "AJ-cal": round(ajc, 5),
                 "fit s": round(timing[name], 1)})

pd.DataFrame(rows).set_index("model")

,concordance,IBS,AJ-cal,fit s
model,,,,
Fine-Gray,0.7521,0.03821,0.01783,0.1
CR-Forest,0.6798,0.03872,0.01464,3.5
DeepHit,0.7247,0.03836,0.03515,3.6
Aalen-Johansen,0.5000,0.03964,0.04267,0.0


## 2.6 Read the table, not just the best number

Two questions to ask of it:

1. Does the model with the best **concordance** also have the best
   **AJ-calibration**? If not, that is sub-question 2 answering itself.
2. Is any model's calibration worse than a model with **no covariates at all**
   would be? That is what the Aalen–Johansen reference is for.

A single fold cannot settle either — the numbers move from fold to fold, which
is why notebook 3 runs all of them and notebook 4 reports spread rather than
point estimates.

## What to take from this notebook

- Scaling and imputation belong **inside the fold**, fitted on training rows only.
- Every model returns a different array layout, and getting it wrong fails silently.
- Component 0 of the packed array is survival, not an event.
- Concordance cannot see calibration; IBS confounds the two; AJ-calibration isolates it.
- One fold is an anecdote. The next notebook runs the grid.